In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime
import time

# ------------------------------------------------------------
# 1. Download SPY data
# ------------------------------------------------------------

ticker = yf.Ticker("SPY")

spot = ticker.fast_info["lastPrice"]

print(f"SPY Spot Price: ${spot:.4f}")

# Get available expiries
expiries = ticker.options

print("Available expiries:")
print(expiries[:10])

SPY Spot Price: $765.7200
Available expiries:
('2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-31', '2026-09-04', '2026-09-11', '2026-09-18', '2026-09-25')


In [2]:
from datetime import datetime

today = datetime.now()

target_expiry = None
min_difference = float("inf")

for expiry in expiries:

    expiry_date = datetime.strptime(expiry, "%Y-%m-%d")

    days_to_expiry = (expiry_date - today).days

    if days_to_expiry > 0:

        difference = abs(days_to_expiry - 30)

        if difference < min_difference:
            min_difference = difference
            target_expiry = expiry

print("Selected expiry:", target_expiry)

expiry_date = datetime.strptime(target_expiry, "%Y-%m-%d")

T = (expiry_date - today).days / 365

print(f"Time to maturity: {T:.6f} years")
print(f"Days to maturity: {T * 365:.0f}")

Selected expiry: 2026-09-25
Time to maturity: 0.087671 years
Days to maturity: 32


In [3]:
option_chain = ticker.option_chain(target_expiry)

calls = option_chain.calls.copy()

print("Number of calls:", len(calls))

display(
    calls[
        [
            "strike",
            "lastPrice",
            "bid",
            "ask",
            "volume",
            "openInterest",
            "impliedVolatility"
        ]
    ].head()
)

Number of calls: 120


,strike,lastPrice,bid,ask,volume,openInterest,impliedVolatility
0,550.0,229.19,215.46,218.98,NaN,1.0,0.588871
1,590.0,185.40,175.66,179.18,2.0,NaN,0.570988
2,600.0,168.55,165.73,169.25,1.0,259.0,0.543828
3,640.0,131.30,126.03,129.55,9.0,766.0,0.436071
4,645.0,136.18,121.08,124.60,NaN,1.0,0.422918


In [4]:
# Remove invalid quotes
liquid_calls = calls[
    (calls["bid"] > 0) &
    (calls["ask"] > 0) &
    (calls["ask"] >= calls["bid"]) &
    (calls["volume"] > 0)
].copy()

# Distance from ATM
liquid_calls["distance_from_spot"] = (
    liquid_calls["strike"] - spot
).abs()

# Select closest liquid strike
selected = liquid_calls.loc[
    liquid_calls["distance_from_spot"].idxmin()
]

print("[Selected Option]")
print(selected)

[Selected Option]
contractSymbol               SPY260925C00766000
lastTradeDate         2026-08-21 20:09:34+00:00
strike                                    766.0
lastPrice                                 12.22
bid                                        12.2
ask                                       12.25
change                                 1.020001
percentChange                          9.107147
volume                                    332.0
openInterest                               64.0
impliedVolatility                      0.132836
inTheMoney                                False
contractSize                            REGULAR
currency                                    USD
distance_from_spot                     0.280029
Name: 43, dtype: object


In [5]:
strike = float(selected["strike"])

bid = float(selected["bid"])
ask = float(selected["ask"])

market_price = (bid + ask) / 2

print(f"Spot:          ${spot:.4f}")
print(f"Strike:        ${strike:.4f}")
print(f"Bid:           ${bid:.4f}")
print(f"Ask:           ${ask:.4f}")
print(f"Market Price:  ${market_price:.4f}")

Spot:          $765.7200
Strike:        $766.0000
Bid:           $12.2000
Ask:           $12.2500
Market Price:  $12.2250


In [6]:
# Download one year of daily SPY data
hist = ticker.history(period="1y")

prices = hist["Close"].dropna()

# Log returns
log_returns = np.log(prices / prices.shift(1)).dropna()

# Annualized historical volatility
historical_vol = log_returns.std() * np.sqrt(252)

print(f"Historical Volatility: {historical_vol * 100:.4f}%")

Historical Volatility: 12.8327%


In [7]:
irx = yf.Ticker("^IRX")

risk_free_rate = irx.fast_info["lastPrice"] / 100

print(f"Risk-free rate: {risk_free_rate * 100:.4f}%")

Risk-free rate: 3.7100%


In [8]:
from Options_architecture import EuropeanOption, OptionType
from black_scholes import BlackScholesPricer

option = EuropeanOption(
    strike=strike,
    maturity=T,
    option_type=OptionType.CALL
)


bs_pricer = BlackScholesPricer(
    spot=spot,
    rate=risk_free_rate,
    volatility=historical_vol
)

bs_price = bs_pricer.price(option)

print(f"Black-Scholes Price: ${bs_price:.6f}")

Black-Scholes Price: $12.726848


In [9]:
from Monte_Carlo_Simulation_Engine import MonteCarloPricer

mc_pricer = MonteCarloPricer(
    spot=spot,
    rate=risk_free_rate,
    volatility=historical_vol
)

start_time = time.time()

mc_price = mc_pricer.price(
    option=option,
    num_paths=100_000,
    num_steps=252,
    antithetic=True,
    seed=42
)

runtime = time.time() - start_time

print(f"Monte Carlo Price:    ${mc_price:.6f}")
print(f"Simulation Runtime:   {runtime:.4f} seconds")

Monte Carlo Price:    $12.641455
Simulation Runtime:   2.7691 seconds


In [10]:
results = pd.DataFrame({
    "Method": [
        "Market",
        "Black-Scholes",
        "Monte Carlo"
    ],
    "Price": [
        market_price,
        bs_price,
        mc_price
    ]
})

display(results)

,Method,Price
0,Market,12.225000
1,Black-Scholes,12.726848
2,Monte Carlo,12.641455


In [11]:
bs_error = bs_price - market_price
mc_error = mc_price - market_price

bs_error_pct = abs(bs_error) / market_price * 100
mc_error_pct = abs(mc_error) / market_price * 100

print("========== MODEL VALIDATION ==========")

print(f"Market Price:       ${market_price:.6f}")

print(f"\nBlack-Scholes:")
print(f"Price:              ${bs_price:.6f}")
print(f"Absolute Error:     ${abs(bs_error):.6f}")
print(f"Percentage Error:   {bs_error_pct:.4f}%")

print(f"\nMonte Carlo:")
print(f"Price:              ${mc_price:.6f}")
print(f"Absolute Error:     ${abs(mc_error):.6f}")
print(f"Percentage Error:   {mc_error_pct:.4f}%")

========== MODEL VALIDATION ==========
Market Price:       $12.225000

Black-Scholes:
Price:              $12.726848
Absolute Error:     $0.501848
Percentage Error:   4.1051%

Monte Carlo:
Price:              $12.641455
Absolute Error:     $0.416455
Percentage Error:   3.4066%


In [12]:
from Implied_Volatility_Calculator import ImpliedVolatilityCalculator

In [19]:
iv_calc = ImpliedVolatilityCalculator(spot=spot, rate=risk_free_rate)

# Calculate implied volatility
market_iv = iv_calc.newton_raphson(option=option, market_price=market_price, initial_guess=historical_vol)

In [20]:
print(f"Historical Volatility: {historical_vol:.4%}")
print(f"Market Implied Vol:    {market_iv:.4%}")

Historical Volatility: 12.8327%
Market Implied Vol:    12.2752%


In [21]:
iv_pricer = BlackScholesPricer(
    spot=spot,
    rate=risk_free_rate,
    volatility=market_iv
)

iv_price = iv_pricer.price(option)

print(f"Market Price: ${market_price:.6f}")
print(f"IV Price:     ${iv_price:.6f}")
print(f"Error:        ${abs(iv_price - market_price):.10f}")

Market Price: $12.225000
IV Price:     $12.225000
Error:        $0.0000000000


In [22]:
vol_comparison = pd.DataFrame({
    "Volatility Measure": [
        "Historical Volatility",
        "Market Implied Volatility"
    ],
    "Volatility": [
        historical_vol,
        market_iv
    ]
})

vol_comparison["Volatility (%)"] = (
    vol_comparison["Volatility"] * 100
)

display(vol_comparison)

,Volatility Measure,Volatility,Volatility (%)
0,Historical Volatility,0.128327,12.832652
1,Market Implied Volatility,0.122752,12.275240


In [23]:
vol_difference = market_iv - historical_vol

print(
    f"Implied - Historical Volatility: "
    f"{vol_difference * 100:.4f} percentage points"
)

print(
    f"Relative Difference: "
    f"{vol_difference / historical_vol * 100:.2f}%"
)

Implied - Historical Volatility: -0.5574 percentage points
Relative Difference: -4.34%
